##2.4: Ingesting FII / DII Flow Data

In [0]:
from pyspark.sql import functions as F

# --- 1. CONFIGURATION ---
fii_path = "/Volumes/iran_israel_capstone_project/bronze/landing_zone/fii_data/Merged_FII_Data.csv"

# --- 2. INGESTION ---
df_fii_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(fii_path)

# --- 3. TRANSFORMATION & MAPPING (DII REMOVED) ---
# We are only selecting the date and the three FII-specific columns.
# No DII columns are defined or selected here.
df_fii_mapped = df_fii_raw.select(
    F.to_date(F.col("DATE")).alias("date"),
    F.col("`FII EQUITY Net Purchase / Sales`").cast("float").alias("fii_net_buy_sell_cr"),
    F.col("`FII EQUITY Gross Purchase`").cast("float").alias("fii_gross_buy_cr"),
    F.col("`FII EQUITY Gross Sales`").cast("float").alias("fii_gross_sell_cr")
)

# --- 4. AUDIT COLUMNS ---
# Required for Bronze KPI Compliance 
df_fii_final = df_fii_mapped.withColumn("ingestion_timestamp", F.current_timestamp()) \
                            .withColumn("source_file", F.lit("Merged_FII_Data.csv"))

# --- 5. SAVE TO BRONZE TABLE ---
# Using overwrite mode to ensure the old schema (with the DII column) is replaced
df_fii_final.write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("iran_israel_capstone_project.bronze.fii_raw")

# --- 6. VERIFICATION ---
print("SUCCESS: Bronze Table 'fii_raw' updated. DII column has been removed.")
display(spark.table("iran_israel_capstone_project.bronze.fii_raw").limit(10))